In [8]:
import duckdb

con = duckdb.connect("../data/market_data.duckdb", read_only=True)

# %% Active trades: breakout triggered in current trend session, still holding
df_active = con.execute("""
    SELECT
        ticker, company_name, sector, industry,
        ROUND(market_cap / 1e9, 2) AS mcap_bn,
        entry_date, ROUND(entry_price, 2) AS entry_price,
        ROUND(close_price, 2) AS current_close,
        ROUND(pct_return, 2) AS pct_return,
        days_held
    FROM v_screener_dashboard
    WHERE status = 'ACTIVE'
    ORDER BY entry_date, ticker
""").fetchdf()
print(f"Active trades: {len(df_active)}")
df_active

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Active trades: 115


,ticker,company_name,sector,industry,mcap_bn,entry_date,entry_price,current_close,pct_return,days_held
0,DSGN,"Design Therapeutics, Inc.",Healthcare,Biotechnology,0.59,2025-11-07,7.56,10.53,39.29,140
1,TYRA,"Tyra Biosciences, Inc.",Healthcare,Biotechnology,2.00,2025-11-20,21.00,36.29,72.81,127
2,FIVE,"Five Below, Inc.",Consumer Cyclical,Discount Stores,12.54,2025-12-04,168.42,221.72,31.65,113
3,ECPG,"Encore Capital Group, Inc.",Financial Services,Financial - Mortgages,1.52,2025-12-10,55.33,68.87,24.47,107
4,ECVT,Ecovyst Inc.,Basic Materials,Chemicals - Specialty,1.33,2025-12-11,9.28,12.84,38.36,106
...,...,...,...,...,...,...,...,...,...,...
110,CRUS,"Cirrus Logic, Inc.",Technology,Semiconductors,6.94,2026-03-26,148.69,143.39,-3.56,1
111,KALV,"KalVista Pharmaceuticals, Inc.",Healthcare,Biotechnology,0.86,2026-03-26,18.95,19.33,2.01,1
112,KOD,Kodiak Sciences Inc.,Healthcare,Biotechnology,1.18,2026-03-26,39.76,37.00,-6.94,1
113,KODK,Eastman Kodak Company,Industrials,Specialty Business Services,0.65,2026-03-26,8.88,9.46,6.53,1


In [9]:
# %% Watchlist: tickers in SEPA trend today but no breakout yet in this session
df_watchlist = con.execute("""
    WITH trend_sessions AS (
        SELECT
            t2.ticker, t2.date, t2.trend_ok, t2.breakout_ok,
            CASE WHEN t2.trend_ok AND NOT COALESCE(
                LAG(t2.trend_ok) OVER (PARTITION BY t2.ticker ORDER BY t2.date),
                FALSE
            ) THEN 1 ELSE 0 END AS session_start
        FROM t2_screener_features t2
        INNER JOIN company_profiles c ON t2.ticker = c.ticker
        WHERE c.is_active = TRUE
    ),
    sessions AS (
        SELECT ticker, date, breakout_ok,
            SUM(session_start) OVER (PARTITION BY ticker ORDER BY date) AS session_id
        FROM trend_sessions WHERE trend_ok
    ),
    current_sessions AS (
        SELECT ticker, session_id
        FROM sessions
        WHERE date = (SELECT MAX(date) FROM t2_screener_features)
    ),
    session_stats AS (
        SELECT s.ticker, s.session_id,
               MIN(s.date) AS session_start_date,
               BOOL_OR(s.breakout_ok) AS had_breakout
        FROM sessions s
        INNER JOIN current_sessions cs
            ON s.ticker = cs.ticker AND s.session_id = cs.session_id
        GROUP BY s.ticker, s.session_id
    )
    SELECT
        ss.ticker,
        cp.name AS company_name,
        cp.sector,
        cp.industry,
        ROUND(cp.market_cap / 1e9, 2) AS mcap_bn,
        ss.session_start_date,
        CAST(datediff('day', ss.session_start_date, p.date) AS INTEGER) AS days_in_trend,
        ROUND(p.close, 2) AS current_close
    FROM session_stats ss
    INNER JOIN company_profiles cp ON ss.ticker = cp.ticker
    INNER JOIN price_data p ON ss.ticker = p.ticker
        AND p.date = (SELECT MAX(date) FROM price_data WHERE ticker = ss.ticker)
    WHERE NOT ss.had_breakout
    ORDER BY ss.session_start_date, ss.ticker
""").fetchdf()
print(f"Watchlist (trend, no breakout yet): {len(df_watchlist)}")
df_watchlist


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Watchlist (trend, no breakout yet): 74


,ticker,company_name,sector,industry,mcap_bn,session_start_date,days_in_trend,current_close
0,STRL,"Sterling Infrastructure, Inc.",Industrials,Engineering & Construction,12.34,2026-01-16,70,420.24
1,INVA,"Innoviva, Inc.",Healthcare,Biotechnology,1.40,2026-02-25,30,22.66
2,EHAB,"Enhabit, Inc.",Healthcare,Medical - Care Facilities,0.69,2026-03-03,24,13.66
3,NKTR,Nektar Therapeutics,Healthcare,Biotechnology,1.47,2026-03-03,24,68.66
4,CIEN,Ciena Corporation,Technology,Communication Equipment,54.31,2026-03-09,18,401.61
...,...,...,...,...,...,...,...,...
69,MRVL,"Marvell Technology, Inc.",Technology,Semiconductors,76.86,2026-03-26,1,94.88
70,ENS,EnerSys,Industrials,Electrical Equipment & Parts,6.10,2026-03-27,0,171.37
71,GSAT,"Globalstar, Inc.",Communication Services,Telecommunications Services,7.55,2026-03-27,0,64.16
72,MOVE,Movano Inc.,Healthcare,Medical - Devices,0.01,2026-03-27,0,18.34


In [10]:
# %% Recent exits (last 30 days)
df_exits = con.execute("""
    SELECT
        ticker, company_name, sector, industry,
        ROUND(market_cap / 1e9, 2) AS mcap_bn,
        entry_date, exit_date,
        ROUND(entry_price, 2) AS entry_price,
        ROUND(close_price, 2) AS exit_price,
        ROUND(pct_return, 2) AS pct_return,
        days_held
    FROM v_screener_dashboard
    WHERE status = 'EXITED'
      AND exit_date >= CURRENT_DATE - INTERVAL '30 days'
    ORDER BY exit_date DESC, pct_return DESC
""").fetchdf()
print(f"Recent exits (last 30 days): {len(df_exits)}")
df_exits

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Recent exits (last 30 days): 497


,ticker,company_name,sector,industry,mcap_bn,entry_date,exit_date,entry_price,exit_price,pct_return,days_held
0,DAWN,"Day One Biopharmaceuticals, Inc.",Healthcare,Biotechnology,2.21,2026-03-04,2026-03-26,12.87,21.40,66.28,22
1,ATI,ATI Inc.,Industrials,Manufacturing - Metal Fabrication,19.42,2025-10-23,2026-03-26,89.55,143.94,60.74,154
2,LASR,"nLIGHT, Inc.",Technology,Semiconductors,3.67,2026-01-08,2026-03-26,40.03,63.86,59.53,77
3,NE,Noble Corporation Plc,Energy,Oil & Gas Drilling,7.44,2026-01-14,2026-03-26,32.58,49.83,52.95,71
4,PR,Permian Resources Corporation,Energy,Oil & Gas Exploration & Production,14.36,2026-01-14,2026-03-26,14.67,21.46,46.28,71
...,...,...,...,...,...,...,...,...,...,...,...
492,TRST,TrustCo Bank Corp NY,Financial Services,Banks - Regional,0.79,2026-02-04,2026-02-26,45.29,44.48,-1.77,22
493,MSBI,"Midland States Bancorp, Inc.",Financial Services,Banks - Regional,0.45,2026-01-22,2026-02-26,23.78,23.03,-3.14,35
494,TNGX,"Tango Therapeutics, Inc.",Healthcare,Biotechnology,2.26,2026-01-07,2026-02-26,11.79,11.40,-3.31,50
495,QCRH,"QCR Holdings, Inc.",Financial Services,Banks - Regional,1.37,2026-02-05,2026-02-26,93.98,89.69,-4.56,21


In [11]:
# %%
con.close()